# 07 — Validazione del server MCP Movie RAG

Questo notebook verifica il contratto `movie-search/v1` senza avviare un sottoprocesso `stdio`. Il client MCP e il server comunicano **in-process**, ma attraversano gli stessi schemi e gli stessi tool usati da una futura LLM.

Obiettivi:

1. controllare i profili tecnici ammessi;
2. verificare che il tool normale non esponga modello o reranker;
3. eseguire una ricerca reale su Qdrant;
4. verificare `no_results` e recupero dettagli;
5. provare opzionalmente un profilo diverso per una singola query.

Il notebook non crea collection, non indicizza film e non modifica Qdrant.

## 1. Prerequisiti

Dalla radice del progetto, prima di aprire il notebook:

```bash
docker compose up -d
python -m pip install -e ".[bge,mcp,test]"
```

La collection deve essere già stata popolata dal notebook 05. Il download di BGE-M3 avviene alla prima query se il modello non è nella cache locale.

In [3]:
from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import JSON, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mcp import Client
from movie_rag.mcp_profiles import load_profile_catalog
from movie_rag.mcp_server import create_mcp_server
from movie_rag.mcp_service import MovieSearchServiceRegistry

print('Project root:', PROJECT_ROOT)

Project root: /Users/danieleraveglia/Documents/uniMi/TESI/progetto


## 2. Configurazione centralizzata

Il profilo normale resta `dense_v1`, coerente con la valutazione svolta. Il tool sperimentale è abilitato soltanto dentro questo notebook per mostrare il confronto per-query. `RUN_RERANK_EXAMPLE=False` evita di caricare il cross-encoder se non serve.

In [15]:
QDRANT_URL = 'http://localhost:6333'
DEFAULT_PROFILE = 'dense_v1'
ENABLE_EXPERIMENTAL_TOOL = True
RUN_RERANK_EXAMPLE = False
PROFILE_CONFIG_PATH = PROJECT_ROOT / 'configs' / 'mcp_profiles_v1.json'

QUERY = 'un film malinconico sul rapporto tra memoria e identità'
FILTERS = {'max_runtime': 120, 'exclude_genres': ['Horror']}
LIMIT = 10

os.environ['QDRANT_URL'] = QDRANT_URL
os.environ['MOVIE_RAG_PROFILE'] = DEFAULT_PROFILE
os.environ['MOVIE_RAG_PROFILES_PATH'] = str(PROFILE_CONFIG_PATH)

## 3. Profili autorizzati

Un profilo è l'unità atomica di configurazione: encoder, collection compatibile, rappresentazione e reranker opzionale. Il client non può passare un nome modello arbitrario.

In [6]:
catalog = load_profile_catalog(PROFILE_CONFIG_PATH)
profile_rows = []
for profile_id, profile in catalog.profiles.items():
    profile_rows.append({
        'profile_id': profile_id,
        'pipeline': profile.pipeline,
        'encoder': profile.dense_config.embedding_model,
        'collection': profile.dense_config.collection_name,
        'reranker': None if profile.reranker_config is None else profile.reranker_config.model_name,
        'default': profile_id == catalog.default_profile_id,
    })
pd.DataFrame(profile_rows)

,profile_id,pipeline,encoder,collection,reranker,default
0,dense_v1,dense,BAAI/bge-m3,movies_d_full_enriched_dense_v1,NaN,True
1,dense_rerank_v1,dense_rerank,BAAI/bge-m3,movies_d_full_enriched_dense_v1,BAAI/bge-reranker-v2-m3,False


## 4. Creazione del server e introspezione del contratto

Il registry connette Qdrant, ma modelli e profili sono lazy: vengono materializzati al primo uso. Il client in-process permette di interrogare `tools/list` come farebbe un host MCP.

In [7]:
registry = MovieSearchServiceRegistry.from_environment(
    profile_id=DEFAULT_PROFILE,
    profiles_path=str(PROFILE_CONFIG_PATH),
)
server = create_mcp_server(
    registry=registry,
    enable_experimental=ENABLE_EXPERIMENTAL_TOOL,
)

async with Client(server, raise_exceptions=True) as client:
    tool_listing = await client.list_tools()

tool_rows = []
for tool in tool_listing.tools:
    tool_rows.append({
        'tool': tool.name,
        'input_fields': sorted(tool.input_schema.get('properties', {})),
        'structured_output': tool.output_schema is not None,
        'read_only': tool.annotations.read_only_hint,
    })
pd.DataFrame(tool_rows)

,tool,input_fields,structured_output,read_only
0,search_movies,"[filters, limit, query]",True,True
1,get_movie_details,[movie_ids],True,True
2,search_movies_experimental,"[execution_overrides, filters, limit, profile_...",True,True


[09/11/26 16:00:12] INFO     HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"              ]8;id=8133969;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8133970;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

In [8]:
production_tool = next(tool for tool in tool_listing.tools if tool.name == 'search_movies')
production_fields = set(production_tool.input_schema['properties'])
forbidden = {
    'profile_id', 'model', 'rerank', 'candidate_pool_size',
    'exact', 'score_threshold', 'collection_name',
}
assert production_fields == {'query', 'filters', 'limit'}
assert production_fields.isdisjoint(forbidden)
print('Contratto production verificato:', sorted(production_fields))

Contratto production verificato: ['filters', 'limit', 'query']


## 5. Ricerca tramite il tool production

Questa è una chiamata MCP reale. Il profilo non compare negli argomenti; viene riportato soltanto nell'output per audit.

In [9]:
async with Client(server, raise_exceptions=True) as client:
    search_call = await client.call_tool(
        'search_movies',
        {'query': QUERY, 'filters': FILTERS, 'limit': LIMIT},
    )

assert not search_call.is_error
search_output = search_call.structured_content
display(JSON(search_output, expanded=False))
pd.DataFrame([
    {
        'rank': item['rank'],
        'dense_rank': item['dense_rank'],
        'movie_id': item['movie_id'],
        'title': item['title'],
        'final_score': item['scores']['final'],
        'dense_score': item['scores']['dense'],
        'reranker_score': item['scores']['reranker'],
    }
    for item in search_output['results']
])

[09/11/26 16:02:30] INFO     HTTP Request: GET                                                      ]8;id=8133975;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8133976;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/exis                
                             ts "HTTP/1.1 200 OK"                                                                  

                    INFO     HTTP Request: GET                                                      ]8;id=8133981;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8133982;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1                     
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST                                                     ]8;id=8133987;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8133988;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/count "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: POST                                                     ]8;id=8133993;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8133994;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/count "HTTP/1.1 200 OK"                                                            

[09/11/26 16:02:36] INFO     HTTP Request: HEAD                                                     ]8;id=8133999;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134000;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1                 
                             307 Temporary Redirect"                                                               

[09/11/26 16:02:37] INFO     HTTP Request: HEAD                                                     ]8;id=8134005;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134006;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b                
                             028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"                          

... (11 outputs left out)

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 28.86it/s]


[09/11/26 16:02:48] INFO     HTTP Request: POST                                                     ]8;id=8134063;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134064;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

<IPython.core.display.JSON object>

,rank,dense_rank,movie_id,title,final_score,dense_score,reranker_score
0,1,1,4226,Memento (2000),0.519201,0.519201,None
1,2,2,1270,Back to the Future (1985),0.467724,0.467724,None
2,3,3,367,"Mask, The (1994)",0.461991,0.461991,None
3,4,4,1580,Men in Black (a.k.a. MIB) (1997),0.444607,0.444607,None
4,5,5,608,Fargo (1996),0.437194,0.437194,None
5,6,6,1198,Raiders of the Lost Ark (Indiana Jones and the...,0.425990,0.425990,None
6,7,7,1,Toy Story (1995),0.419718,0.419718,None
7,8,8,4306,Shrek (2001),0.406583,0.406583,None
8,9,9,364,"Lion King, The (1994)",0.406330,0.406330,None
9,10,10,50,"Usual Suspects, The (1995)",0.404454,0.404454,None


## 6. Caso valido senza risultati

Una lingua non presente nel catalogo deve produrre `status=no_results` e `results=[]`, non interrompere la pipeline. Il retriever non rilassa automaticamente il filtro.

In [10]:
async with Client(server, raise_exceptions=True) as client:
    empty_call = await client.call_tool(
        'search_movies',
        {
            'query': 'un film qualunque',
            'filters': {'languages': ['zz']},
            'limit': 10,
        },
    )

empty_output = empty_call.structured_content
assert empty_output['status'] == 'no_results'
assert empty_output['results'] == []
display(JSON(empty_output, expanded=False))

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]


[09/11/26 16:04:20] INFO     HTTP Request: POST                                                     ]8;id=8134069;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134070;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

<IPython.core.display.JSON object>

## 7. Dettagli grounded

Richiediamo i dettagli soltanto degli ID recuperati. Gli ID mancanti vengono dichiarati esplicitamente.

In [11]:
movie_ids = [item['movie_id'] for item in search_output['results'][:3]]
if movie_ids:
    async with Client(server, raise_exceptions=True) as client:
        details_call = await client.call_tool(
            'get_movie_details', {'movie_ids': movie_ids}
        )
    details_output = details_call.structured_content
    display(JSON(details_output, expanded=False))
else:
    print('La ricerca non ha restituito ID: dettaglio non eseguito.')

[09/11/26 16:04:47] INFO     HTTP Request: POST                                                     ]8;id=8134075;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134076;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts "HTTP/1.1 200 OK"                                                                  

<IPython.core.display.JSON object>

## 8. Override sperimentale per singola query

La cella seguente è opzionale. Se attivata, seleziona `dense_rerank_v1` soltanto per questa chiamata. La chiamata production successiva continua a usare `dense_v1`.

In [14]:
if RUN_RERANK_EXAMPLE:
    async with Client(server, raise_exceptions=True) as client:
        rerank_call = await client.call_tool(
            'search_movies_experimental',
            {
                'query': QUERY,
                'profile_id': 'dense_rerank_v1',
                'filters': FILTERS,
                'limit': LIMIT,
                'execution_overrides': {'candidate_pool_size': 20},
            },
        )
        default_again = await client.call_tool(
            'search_movies',
            {'query': QUERY, 'filters': FILTERS, 'limit': LIMIT},
        )
    rerank_output = rerank_call.structured_content
    default_output = default_again.structured_content
    assert rerank_output['pipeline']['profile_id'] == 'dense_rerank_v1'
    assert default_output['pipeline']['profile_id'] == 'dense_v1'
    display(pd.DataFrame([
        {
            'call': 'experimental',
            'profile': rerank_output['pipeline']['profile_id'],
            'titles': [item['title'] for item in rerank_output['results']],
        },
        {
            'call': 'production after experiment',
            'profile': default_output['pipeline']['profile_id'],
            'titles': [item['title'] for item in default_output['results']],
        },
    ]))
else:
    print('Esempio reranker non eseguito. Impostare RUN_RERANK_EXAMPLE=True per abilitarlo.')

[09/11/26 16:06:46] INFO     HTTP Request: POST                                                     ]8;id=8134081;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134082;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

                    INFO     HTTP Request: HEAD                                                     ]8;id=8134087;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134088;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/config.jso                
                             n "HTTP/1.1 307 Temporary Redirect"                                                   

                    INFO     HTTP Request: HEAD                                                     ]8;id=8134093;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134094;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m                
                             3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/config.json "HTTP/1.1 200                  
                             OK"                                                                                   

[09/11/26 16:06:47] INFO     HTTP Request: HEAD                                                     ]8;id=8134099;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134100;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/BAAI/bge-reranker-v2-m3/resolve/main/tokenizer_                
                             config.json "HTTP/1.1 307 Temporary Redirect"                                         

                    INFO     HTTP Request: HEAD                                                     ]8;id=8134105;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134106;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m                
                             3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/tokenizer_config.json                      
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: GET                                                      ]8;id=8134111;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134112;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3/tree/main/ad                
                             ditional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not                
                             Found"                                                                                

... (5 outputs left out)

                    INFO     HTTP Request: HEAD                                                     ]8;id=8134147;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134148;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-v2-m                
                             3/953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e/config.json "HTTP/1.1 200                  
                             OK"                                                                                   

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[09/11/26 16:06:58] INFO     HTTP Request: POST                                                     ]8;id=8134153;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8134154;file:///Users/danieleraveglia/Documents/uniMi/TESI/progetto/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:6333/collections/movies_d_full_enriched_dense_v1/poin                
                             ts/query "HTTP/1.1 200 OK"                                                            

,call,profile,titles
0,experimental,dense_rerank_v1,"[Beauty and the Beast (1991), Memento (2000), ..."
1,production after experiment,dense_v1,"[Memento (2000), Back to the Future (1985), Ma..."


## 9. Avvio come server locale

Dopo la validazione in-process, il server può essere avviato in `stdio` dalla radice del progetto:

```bash
export QDRANT_URL=http://localhost:6333
export MOVIE_RAG_PROFILE=dense_v1
movie-rag-mcp
```

Per ispezionarlo con il client di sviluppo MCP, avviare l'Inspector puntando allo stesso comando. Non aggiungere `print` al server: con `stdio`, stdout è riservato al protocollo.

Il prossimo passaggio sarà collegare una LLM al solo tool production e verificare parsing dei filtri, gestione di `no_results` e risposta grounded.